# 06 — Tensor methods (order 3)

Compare cubic (order 2) vs approximate tensor (order 3) methods. Track iterations vs oracle cost (Hessian evaluations).

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
from cubic_reg.problems import Quadratic, Rosenbrock, LogSumExp
from cubic_reg.solvers import cr, tensor
from cubic_reg.plotting import plot_optimality_gap

%matplotlib inline

In [ ]:
p = Quadratic(n=20, condition=50.0, seed=0)
x0 = np.ones(p.dim)
r_cr = cr.minimize(p, x0=x0, M=1.0, eps=1e-8, max_iter=40)
r_t = tensor.minimize(p, x0=x0, L=1.0, eps=1e-8, max_iter=40)
print("CR    ", r_cr.nit, r_cr.n_hess, r_cr.time_sec, r_cr.grad_norm)
print("Tensor", r_t.nit, r_t.n_hess, r_t.time_sec, r_t.grad_norm)
plot_optimality_gap({"CR": r_cr, "Tensor": r_t}, f_star=p.f_star)
plt.show()

In [ ]:
# Break-even exploration: cost per iteration grows with n for tensor (FD on Hessian)
rows = []
for n in [8, 12, 16, 24]:
    p = LogSumExp(n=n, m=3*n, seed=0)
    x0 = 0.1 * np.ones(n)
    r_cr = cr.minimize(p, x0=x0, M=1.0, eps=1e-5, max_iter=25)
    r_t = tensor.minimize(p, x0=x0, L=1.0, eps=1e-5, max_iter=15)
    rows.append((n, r_cr.time_sec/max(r_cr.nit,1), r_t.time_sec/max(r_t.nit,1), r_cr.nit, r_t.nit))
print(f"{'n':>4} {'CR s/it':>10} {'Tensor s/it':>12} {'CR nit':>8} {'T nit':>8}")
for row in rows:
    print(f"{row[0]:4d} {row[1]:10.4f} {row[2]:12.4f} {row[3]:8d} {row[4]:8d}")
ns = [r[0] for r in rows]
plt.figure(figsize=(6, 3.5))
plt.plot(ns, [r[1] for r in rows], "o-", label="CR")
plt.plot(ns, [r[2] for r in rows], "s-", label="Tensor")
plt.xlabel("n"); plt.ylabel("sec / iter"); plt.legend(); plt.title("Break-even cost")
plt.grid(True, alpha=0.3); plt.show()